In [54]:
d_model = 64        # hidden size
history_len = 30
forecast_len = 7


In [55]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GRN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, output_dim)
        self.fc2 = nn.Linear(output_dim, output_dim)

        self.gate = nn.Linear(output_dim, output_dim)
        self.layer_norm = nn.LayerNorm(output_dim)

        self.project_residual = (
            nn.Linear(input_dim, output_dim)
            if input_dim != output_dim else None
        )

    def forward(self, x):
        residual = x if self.project_residual is None else self.project_residual(x)

        x = F.elu(self.fc1(x))
        x = self.fc2(x)

        gate = torch.sigmoid(self.gate(x))
        x = gate * x + (1 - gate) * residual

        return self.layer_norm(x)

        x = gate * x + (1 - gate) * residual

        return self.layer_norm(x)


In [56]:
class VariableSelectionNetwork(nn.Module):
    def __init__(self, num_features, d_model):
        super().__init__()

        self.num_features = num_features

        # One GRN per feature
        self.feature_grns = nn.ModuleList([
            GRN(d_model, d_model) for _ in range(num_features)
        ])

        # Weight generator
        self.weight_grn = GRN(d_model * num_features, d_model)
        self.weight_fc = nn.Linear(d_model, num_features)

    def forward(self, x):
        """
        x: [B, T, num_features, d_model]
        """
        B, T, F, D = x.shape

        # Transform each feature
        transformed = []
        for i in range(F):
            transformed.append(self.feature_grns[i](x[:, :, i, :]))

        transformed = torch.stack(transformed, dim=2)  # [B, T, F, D]

        # Compute weights
        flat = transformed.reshape(B, T, F * D)
        weights = self.weight_grn(flat)
        weights = self.weight_fc(weights)
        weights = torch.softmax(weights, dim=-1)  # [B, T, F]

        # Weighted sum
        output = torch.sum(transformed * weights.unsqueeze(-1), dim=2)

        return output


In [57]:
class StaticEncoder(nn.Module):
    def __init__(self, input_dim, d_model):
        super().__init__()
        self.encoder = nn.Linear(input_dim, d_model)

    def forward(self, x):
        return self.encoder(x)


In [58]:
class FeatureEncoder(nn.Module):
    def __init__(self, num_features, d_model):
        super().__init__()
        self.encoders = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(num_features)
        ])

    def forward(self, x):
        """
        x: [B, T, F]
        returns: [B, T, F, d_model]
        """
        outs = []
        for i, enc in enumerate(self.encoders):
            outs.append(enc(x[:, :, i:i+1]))
        return torch.stack(outs, dim=2)


In [59]:
class FeatureEncoder(nn.Module):
    def __init__(self, num_features, d_model):
        super().__init__()
        self.encoders = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(num_features)
        ])

    def forward(self, x):
        """
        x: [B, T, F]
        returns: [B, T, F, d_model]
        """
        outs = []
        for i, enc in enumerate(self.encoders):
            outs.append(enc(x[:, :, i:i+1]))
        return torch.stack(outs, dim=2)


In [60]:
class MinimalTFT(nn.Module):
    def __init__(self,
                 static_dim,
                 past_dim,
                 future_dim,
                 d_model=64,
                 num_heads=4):
        super().__init__()

        # Encoders
        self.static_encoder = StaticEncoder(static_dim, d_model)
        self.past_encoder = FeatureEncoder(past_dim, d_model)
        self.future_encoder = FeatureEncoder(future_dim, d_model)

        # Variable selection
        self.past_vsn = VariableSelectionNetwork(past_dim, d_model)
        self.future_vsn = VariableSelectionNetwork(future_dim, d_model)

        # Attention
        self.temporal_attn = TemporalAttention(d_model, num_heads)

        # Output head
        self.output_layer = nn.Linear(d_model, 1)

    def forward(self, static, past, future):
        """
        static: [B, S]
        past:   [B, T_hist, O]
        future: [B, T_hist + T_fut, K]
        """

        B, T_hist, O = past.shape
        T_total = future.shape[1]

        # Encode
        static_emb = self.static_encoder(static)

        past_emb = self.past_encoder(past)
        future_emb = self.future_encoder(future)


        past_sel = self.past_vsn(past_emb)
        future_sel = self.future_vsn(future_emb)

        # Combine timeline
        temporal_input = torch.cat([past_sel, future_sel[:, T_hist:]], dim=1)

        # Attention
        attn_out = self.temporal_attn(temporal_input)

        # Forecast
        forecast = self.output_layer(attn_out[:, -forecast_len:])
        return forecast.squeeze(-1)


In [61]:
B = 4
static = torch.randn(B, 5)
past = torch.randn(B, 30, 1)
future = torch.randn(B, 37, 4)

model = MinimalTFT(5, 1, 4)
out = model(static, past, future)

print(out.shape)  # [B, 7]


torch.Size([4, 7])


Raw Inputs
[B, T, F]
    ->
FeatureEncoder
[B, T, F, D]
    ->
VariableSelectionNetwork
[B, T, D]
    ->
TemporalAttention
[B, T, D]
    ->
Output Head
[B, T_fut]
